# 多模态RAG-技术方案2

In [1]:
import fitz  # PyMuPDF
from PIL import Image
import io
import os
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

### 抽取文本和图片

In [2]:
text_data = []
img_data = []

In [3]:
with fitz.open('transformer.pdf') as pdf_file:
    # 创建目录
    if not os.path.exists("extracted_images"):
        os.makedirs("extracted_images")

    # 对PDF里面的每一页分别处理
    for page_number in range(len(pdf_file)):
        page = pdf_file[page_number]
        
        # 抽取文本
        text = page.get_text().strip()
        text_data.append({"response": text, "name": page_number+1})
        # 抽取图片
        images = page.get_images(full=True)

        # 保存每一张图片
        for image_index, img in enumerate(images, start=0):
            xref = img[0]  # Get the XREF of the image
            base_image = pdf_file.extract_image(xref)  # 提取图片
            image_bytes = base_image["image"]  # 得到图片的bytes
            image_ext = base_image["ext"]  # 图片后缀
            
            # save
            image = Image.open(io.BytesIO(image_bytes)) # 转 PIL Image对象
            image.save(f"extracted_images/image_{page_number+1}_{image_index+1}.{image_ext}")

In [4]:
len(text_data)

15

In [5]:
text_data[0]

{'response': 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing wi

In [6]:
# !pip install google-genai

from google import genai
from google.genai import types
from PIL import Image
import os

google_client = genai.Client(
    api_key=os.getenv("API_KEY"),
    http_options={
        "base_url": "https://api.zhizengzeng.com/google"
    },
)

### 图片理解（Image Caption）

In [7]:
for img in os.listdir("extracted_images"):
    if img.endswith("png"):
        image = Image.open(f"extracted_images/{img}")
        response = google_client.models.generate_content(
            model="gemini-2.5-flash",  # 可以替换
            contents=[
                "您是一名助理，负责对表格、图像和文本进行摘要，以便于检索。这些摘要将被嵌入并用于检索原始文本或表格元素。请提供简洁明了、便于检索的表格或文本摘要。表格、文本或图像：", image
            ]
        )
        img_data.append({"response": response.text, "name": img})

In [8]:
img_data

[{'response': 'This image illustrates the **Multi-Head Attention mechanism**.\n\nIt depicts the process where inputs Q (Query), K (Key), and V (Value) are first transformed by multiple parallel **Linear** layers. These transformed outputs then feed into `h` parallel **Scaled Dot-Product Attention** blocks. The results from these attention heads are subsequently combined via a **Concatenation** layer, and finally projected through another **Linear** layer to produce the output.',
  'name': 'image_4_2.png'},
 {'response': '该图像描绘了**缩放点积注意力机制**（Scaled Dot-Product Attention）的计算流程图。\n\n**关键点：**\n*   **输入：** 接收 Query (Q)、Key (K) 和 Value (V) 三个输入。\n*   **处理步骤：**\n    1.  将 Q 和 K 进行矩阵乘法 (MatMul)。\n    2.  对结果进行缩放 (Scale)。\n    3.  选择性地应用掩码 (Mask (opt.))。\n    4.  通过 SoftMax 函数进行激活。\n    5.  将 SoftMax 的输出与 V 进行第二次矩阵乘法 (MatMul)。\n*   **输出：** 产生注意力机制的最终输出。\n*   **上下文：** 这是 Transformer 模型中的核心组件，用于计算输入序列中的注意力权重。',
  'name': 'image_4_1.png'},
 {'response': 'This image displays the **Transformer model

### Vectostore

In [9]:
from openai import OpenAI
from langchain_core.embeddings import Embeddings

# 创建 OpenAI 客户端 (用于 Embeddings)
client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
)

class TextEmbeddings(Embeddings):
    """DashScope 兼容的 Embeddings 封装。"""

    def __init__(self, model: str = "text-embedding-v4", dimensions: int = 1024):
        self.model = model
        self.dimensions = dimensions

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        vectors: list[list[float]] = []
        for i in range(0, len(texts), 10):
            chunk = texts[i : i + 10]
            response = client.embeddings.create(
                model=self.model,
                input=chunk,
                dimensions=self.dimensions,
            )
            vectors.extend([item.embedding for item in response.data])
        return vectors

    def embed_query(self, text: str) -> list[float]:
        response = client.embeddings.create(
            model=self.model,
            input=[text],
            dimensions=self.dimensions,
        )
        return response.data[0].embedding

## 处理文档

In [10]:
# ! pip install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Embedding模型
embedding_model = TextEmbeddings()

# 加载文档
docs_list = [Document(page_content=text['response'], metadata={"name": text['name']}) for text in text_data]
img_list = [Document(page_content=img['response'], metadata={"name": img['name']}) for img in img_data]

# 递归切分
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=400, chunk_overlap=50
)

doc_splits = text_splitter.split_documents(docs_list)
img_splits = text_splitter.split_documents(img_list)

In [11]:
len(doc_splits), len(img_splits)

(36, 4)

In [12]:
doc_splits[0]

Document(metadata={'name': 1}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on atte

In [16]:
img_splits[3]

Document(metadata={'name': 'image_3_1.png'}, page_content='This image displays the **Transformer model architecture**, a neural network primarily used for sequence-to-sequence tasks. It consists of an **encoder** and a **decoder**.\n\n**Key Components:**\n*   **Input Embedding** and **Output Embedding**: Convert input/output tokens into vector representations.\n*   **Positional Encoding**: Adds positional information to embeddings.\n*   **Encoder Stack (N layers)**: Each layer includes a **Multi-Head Attention** mechanism and a **Feed Forward** network, both followed by **Add & Norm** (residual connection + layer normalization).\n*   **Decoder Stack (N layers)**: Each layer contains a **Masked Multi-Head Attention** (for self-attention), a **Multi-Head Attention** (cross-attention to encoder outputs), and a **Feed Forward** network, all followed by **Add & Norm**.\n*   **Linear Layer** and **Softmax**: Produce final **Output Probabilities**.\n\nThe diagram illustrates the flow from "In

In [17]:
# ! pip install langchain-chroma
from langchain_chroma import Chroma # faiss, milvus

# 加入到chroma构建index
vectorstore = Chroma.from_documents(
    documents=doc_splits + img_splits, # adding the both text and image splits
    collection_name="multi_model_rag",
    embedding=embedding_model,
)

retriever = vectorstore.as_retriever(
                search_type="similarity",
                search_kwargs={'k': 1}, # number of documents to retrieve
            )

### 检索文档

In [18]:
query = "Transformer（base model）的BLEU得分是多少？"

In [19]:
docs = retriever.invoke(query)

In [20]:
print(docs)

[Document(metadata={'name': 8}, page_content='Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the\nEnglish-to-German and English-to-French newstest2014 tests at a fraction of the training cost.\nModel\nBLEU\nTraining Cost (FLOPs)\nEN-DE\nEN-FR\nEN-DE\nEN-FR\nByteNet [18]\n23.75\nDeep-Att + PosUnk [39]\n39.2\n1.0 · 1020\nGNMT + RL [38]\n24.6\n39.92\n2.3 · 1019\n1.4 · 1020\nConvS2S [9]\n25.16\n40.46\n9.6 · 1018\n1.5 · 1020\nMoE [32]\n26.03\n40.56\n2.0 · 1019\n1.2 · 1020\nDeep-Att + PosUnk Ensemble [39]\n40.4\n8.0 · 1020\nGNMT + RL Ensemble [38]\n26.30\n41.16\n1.8 · 1020\n1.1 · 1021\nConvS2S Ensemble [9]\n26.36\n41.29\n7.7 · 1019\n1.2 · 1021\nTransformer (base model)\n27.3\n38.1\n3.3 · 1018\nTransformer (big)\n28.4\n41.8\n2.3 · 1019\nResidual Dropout\nWe apply dropout [33] to the output of each sub-layer, before it is added to the\nsub-layer input and normalized. In addition, we apply dropout to the sums of the embeddings and the')]


### 输出

In [21]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Prompt
system = """你是问答任务的助理。请根据提供的知识回答问题。答案务必简洁明了。"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved documents: \n\n <docs>{documents}</docs> \n\n User question: <question>{question}</question>"),
    ]
)

# LLM
llm = ChatOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
    model="qwen3-max",
    temperature=0,
)

# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
generation = rag_chain.invoke({"documents":docs[0].page_content, "question": query})
print(generation)

Transformer（base model）的BLEU得分是：  
- 英语到德语（EN-DE）：27.3  
- 英语到法语（EN-FR）：38.1


![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--multi-model-rag-with-captioning)